In [ ]:
# ── BLOCK 1 ──────────────────────────────────────
# Title: Verify Setup
# Purpose: Confirm Python version and libraries work

import sys
print(f"Python version: {sys.version}")

import pandas as pd
import numpy as np
import tensorflow as tf
import sklearn

print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")
print(f"TensorFlow: {tf.__version__}")
print(f"Scikit-learn: {sklearn.__version__}")
print("\nAll libraries loaded. Setup is correct.")

In [ ]:
# ── BLOCK 1 ──────────────────────────────────────
# Title: Imports
# Purpose: Load libraries needed for feature extraction

import ast
import re

print("Imports loaded.")


In [ ]:
# %%
# ── BLOCK 2 ──────────────────────────────────────
# Title: Feature Extractor — Regex + AST combined
# Purpose: Extract 12 features from Python code
# Features 1-10: Regex based (fast, pattern matching)
# Features 11-12: AST based (structural, more accurate)
# v2: Fixed F2 to catch inline keyword args e.g. Client(api_key="x")
#     Fixed F12 to catch keyword argument hardcoded secrets in AST

import ast
import re

def extract_features(code: str) -> list:

    try:
        tree = ast.parse(code)
    except SyntaxError:
        return [0] * 12

    # ── REGEX FEATURES ────────────────────────────

    # F1: SQL injection — string concat near SQL keyword
    f1 = int(bool(
        re.search(r'["\'].*SELECT.*["\'].*\+', code, re.IGNORECASE) or
        re.search(r'\+.*["\'].*SELECT', code, re.IGNORECASE) or
        re.search(r'["\'].*WHERE.*["\'].*\+', code, re.IGNORECASE)
    ))

    # F2: Hardcoded secret — variable assignment OR keyword argument
    # Catches: password = "x"  AND  Client(api_key="x")
    sensitive = ['password', 'secret', 'api_key', 'token', 'pwd', 'key']
    f2 = int(
        any(re.search(rf'{name}\s*=\s*["\']', code, re.IGNORECASE)
            for name in sensitive)
        or
        any(re.search(rf'{name}=["\'][^"\']+["\']', code, re.IGNORECASE)
            for name in sensitive)
    )

    # F3: Insecure eval or exec call
    f3 = int(bool(
        re.search(r'\beval\s*\(', code) or
        re.search(r'\bexec\s*\(', code)
    ))

    # F4: Path traversal — open() with concat OR path variable built from concat
    f4 = int(bool(
        re.search(r'open\s*\([^)]*\+', code) or
        re.search(r'=\s*[\'"][/\\][^\'"]*[\'"]\s*\+', code)
    ))

    # F5: Command injection — os.system concat or shell=True
    f5 = int(bool(
        re.search(r'os\.system\s*\(', code) or
        re.search(r'os\.popen\s*\(', code) or
        re.search(r'subprocess.*shell\s*=\s*True', code)
    ))

    # F6: AST node count — proxy for code complexity
    f6 = sum(1 for _ in ast.walk(tree))

    # F7: Number of string literals
    f7 = len(re.findall(r'["\'][^"\']*["\']', code))

    # F8: Uses os.environ — SAFE signal
    f8 = int('os.environ' in code)

    # F9: Uses parameterized query — SAFE signal
    f9 = int(bool(
        re.search(r'execute\s*\(.*[?%]', code) or
        re.search(r'execute\s*\(.*,\s*\(', code)
    ))

    # F10: Has user input reference
    f10 = int(bool(
        re.search(r'\b(input|request|user_input|form|args|get_json)\b', code)
    ))

    # ── AST FEATURES ──────────────────────────────

    # F11: Count dangerous function calls
    dangerous_sinks = {'eval', 'exec', 'system', 'popen', 'execute'}
    f11 = 0
    for node in ast.walk(tree):
        if isinstance(node, ast.Call):
            func_name = ""
            if isinstance(node.func, ast.Name):
                func_name = node.func.id
            elif isinstance(node.func, ast.Attribute):
                func_name = node.func.attr
            if func_name in dangerous_sinks:
                f11 += 1

    # F12: Hardcoded string to sensitive variable OR keyword argument
    # Catches: password = "x"  AND  Client(api_key="x")
    sensitive_names = {'password', 'secret', 'api_key', 'token', 'pwd', 'key'}
    f12 = 0
    for node in ast.walk(tree):
        # Direct assignment: password = "hardcoded"
        if isinstance(node, ast.Assign):
            for target in node.targets:
                if isinstance(target, ast.Name):
                    if target.id.lower() in sensitive_names:
                        if isinstance(node.value, ast.Constant):
                            if isinstance(node.value.value, str):
                                f12 += 1

        # Keyword argument: Client(api_key="hardcoded")
        if isinstance(node, ast.Call):
            for kw in node.keywords:
                if kw.arg and kw.arg.lower() in sensitive_names:
                    if isinstance(kw.value, ast.Constant):
                        if isinstance(kw.value.value, str):
                            f12 += 1

    return [f1, f2, f3, f4, f5, f6, f7, f8, f9, f10, f11, f12]


print("Feature extractor defined.")
print(f"Returns {len(extract_features('x = 1'))} features per code sample.")

# Quick sanity check on the fixes
test_inline = 'client = Client(api_key="hardcoded_key_xyz")'
test_assign = 'password = "admin123"'
test_safe   = 'api_key = os.environ.get("API_KEY")'

feats_inline = extract_features(test_inline)
feats_assign = extract_features(test_assign)
feats_safe   = extract_features(test_safe)

print(f"\nSanity checks:")
print(f"  Client(api_key='hardcoded') → f2={feats_inline[1]}, f12={feats_inline[11]}  (expect 1, >0)")
print(f"  password = 'admin123'       → f2={feats_assign[1]}, f12={feats_assign[11]}  (expect 1, 1)")
print(f"  api_key = os.environ.get()  → f2={feats_safe[1]},   f12={feats_safe[11]}    (expect 0, 0)")

In [ ]:
# ── BLOCK 3 ──────────────────────────────────────
# Title: Load and combine all 5 datasets
# Purpose: Load all JSON files, combine into one
#          DataFrame, verify counts and quality

import pandas as pd
import json
import os

# ── Hardcoded path for notebook ───────────────────
data_dir = r"E:\Portfolio Project 2026\securescope-ai\backend\data\custom"

print(f"Loading from: {data_dir}")

# ── Load all 5 JSON files ─────────────────────────
files = [
    'sql_injection.json',
    'hardcoded_secrets.json',
    'insecure_eval.json',
    'path_traversal.json',
    'command_injection.json',
]

all_examples = []
for fname in files:
    fpath = os.path.join(data_dir, fname)
    with open(fpath, 'r') as f:
        examples = json.load(f)
        all_examples.extend(examples)
    print(f"  {fname}: {len(examples)} examples")

# ── Convert to DataFrame ──────────────────────────
df = pd.DataFrame(all_examples)

print(f"\n{'='*40}")
print(f"COMBINED DATASET SUMMARY")
print(f"{'='*40}")
print(f"Total rows:           {len(df)}")
print(f"Vulnerable (label=1): {(df['label']==1).sum()}")
print(f"Safe       (label=0): {(df['label']==0).sum()}")
print(f"Columns:              {list(df.columns)}")
print(f"\nType distribution:")
print(df['type'].value_counts().to_string())
print(f"\nSample code preview:")
print(df['code'].iloc[0][:120])

In [ ]:
# ── BLOCK 4 ──────────────────────────────────────
# Title: Extract features from all examples
# Purpose: Run extract_features() on every code
#          sample, build feature matrix, verify
#          features are firing correctly

# ── Run feature extraction ────────────────────────
print("Extracting features from 250 examples...")

feature_rows = []
failed = 0

for i, row in df.iterrows():
    features = extract_features(row['code'])
    feature_rows.append(features)
    if sum(features) == 0:
        failed += 1

# ── Build feature DataFrame ───────────────────────
feature_names = [
    'f1_sql_concat',
    'f2_hardcoded_secret',
    'f3_eval_exec',
    'f4_path_traversal',
    'f5_cmd_injection',
    'f6_func_length',
    'f7_string_count',
    'f8_uses_environ',
    'f9_parameterized',
    'f10_user_input',
    'f11_ast_dangerous_calls',
    'f12_ast_hardcoded_assign'
]

X = pd.DataFrame(feature_rows, columns=feature_names)
y = df['label'].values

print(f"Feature matrix shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"All-zero rows: {failed} out of {len(df)}")

print(f"\nFeature firing rates (% of examples where feature > 0):")
for col in feature_names:
    rate = (X[col] > 0).mean() * 100
    print(f"  {col:<25}: {rate:.1f}%")

print(f"\nSample — first vulnerable example features:")
first_vuln = X[y == 1].iloc[0]
print(first_vuln.to_string())

print(f"\nSample — first safe example features:")
first_safe = X[y == 0].iloc[0]
print(first_safe.to_string())


In [ ]:
# ── BLOCK 5 ──────────────────────────────────────
# Title: Save feature matrix to CSV
# Purpose: Save X (features) and y (labels) as one
#          CSV file to data/processed/features.csv

# ── Combine features and labels ───────────────────
df_features = X.copy()
df_features['label'] = y

# ── Save to processed folder ──────────────────────
processed_dir = r"E:\Portfolio Project 2026\securescope-ai\backend\data\processed"
os.makedirs(processed_dir, exist_ok=True)

save_path = os.path.join(processed_dir, 'features.csv')
df_features.to_csv(save_path, index=False)

print(f"Saved: {save_path}")
print(f"Shape: {df_features.shape}")
print(f"\nFirst 3 rows:")
print(df_features.head(3).to_string())
print(f"\nLabel distribution:")
print(df_features['label'].value_counts().to_string())


In [ ]:
# ── BLOCK 6 ──────────────────────────────────────
# Title: Train ANN classifier
# Purpose: Load features.csv, normalize, split,
#          build ANN, train, evaluate

from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import joblib

# ── Load features.csv ─────────────────────────────
csv_path = r"E:\Portfolio Project 2026\securescope-ai\backend\data\processed\features.csv"
df_train = pd.read_csv(csv_path)

X = df_train.drop('label', axis=1).values
y = df_train['label'].values

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

# ── Normalize features ────────────────────────────
# f6 (length) and f7 (string count) can be large numbers
# StandardScaler brings all features to same scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ── Train / Val / Test split ──────────────────────
# 70% train, 15% val, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X_scaled, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"\nTrain: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

# ── Build ANN ─────────────────────────────────────
model = keras.Sequential([
    keras.layers.Dense(
        128, activation='relu',
        kernel_initializer='he_normal',
        input_shape=(X_train.shape[1],)
    ),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(
        64, activation='relu',
        kernel_initializer='he_normal'
    ),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(
        32, activation='relu',
        kernel_initializer='he_normal'
    ),
    keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        keras.metrics.AUC(name='auc'),
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall')
    ]
)

model.summary()

# ── Callbacks ─────────────────────────────────────
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=7,
        verbose=1
    ),
]

# ── Train ─────────────────────────────────────────
print("\nTraining...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=150,
    batch_size=16,
    callbacks=callbacks,
    verbose=1
)

# ── Evaluate on test set ──────────────────────────
print("\n" + "="*40)
print("TEST SET RESULTS")
print("="*40)
y_pred = (model.predict(X_test) > 0.5).astype(int)
print(classification_report(
    y_test, y_pred,
    target_names=['Safe', 'Vulnerable']
))

# ── Save model and scaler ─────────────────────────
models_dir = r"E:\Portfolio Project 2026\securescope-ai\backend\models\saved"
os.makedirs(models_dir, exist_ok=True)

model.save(os.path.join(models_dir, 'ann_v1.h5'))
joblib.dump(scaler, os.path.join(models_dir, 'scaler.pkl'))

print(f"Model saved: {models_dir}\\ann_v1.h5")
print(f"Scaler saved: {models_dir}\\scaler.pkl")

In [ ]:
model.save(os.path.join(models_dir, 'ann_v1.keras'))

In [ ]:
print(f"df shape before extraction: {df.shape}")
print(f"Unique examples: {df['code'].nunique()}")

In [ ]:
# ── Quick manual test ─────────────────────────────
test_cases = [
    # Should be VULNERABLE
    ('def get_user(uid):\n    return db.execute("SELECT * FROM users WHERE id = " + uid)', 'SQL injection'),
    ('def connect():\n    password = "admin123"\n    return db.connect(password=password)', 'Hardcoded secret'),
    # Should be SAFE
    ('def get_user(uid):\n    return db.execute("SELECT * FROM users WHERE id = ?", (uid,))', 'Parameterized query'),
]

print("=== MANUAL PREDICTION TEST ===\n")
for code, description in test_cases:
    features = extract_features(code)
    features_scaled = scaler.transform([features])
    prob = model.predict(features_scaled, verbose=0)[0][0]
    prediction = "VULNERABLE" if prob > 0.5 else "SAFE"
    print(f"Test: {description}")
    print(f"Prediction: {prediction} (confidence: {prob:.3f})")
    print()

In [ ]:
# ── CVEfixes Block 1 ─────────────────────────────
# Title: Load CVEfixes from HuggingFace
# Purpose: Load real vulnerability dataset

from datasets import load_dataset

print("Loading CVEfixes...")
dataset = load_dataset("DetectVul/CVEFixes")
print(dataset)
print(f"\nTrain: {len(dataset['train'])} examples")
print(f"Test:  {len(dataset['test'])} examples")
print(f"\nFeatures: {dataset['train'].features}")

In [ ]:
# ── CVEfixes Block 2 ─────────────────────────────
# Title: Inspect Dataset Structure
# Purpose: Understand exactly what each example
#          looks like before processing

import pandas as pd

train_data = dataset['train']

# Look at first 5 examples carefully
print("=== STRUCTURE INSPECTION ===\n")
for i in range(3):
    ex = train_data[i]
    func_label = int(any(l == 1 for l in ex['label']))
    vuln_types = list(set(t for t, l in zip(ex['type'], ex['label']) if l == 1))
    
    print(f"Example {i}:")
    print(f"  Lines count:     {len(ex['raw_lines'])}")
    print(f"  Function label:  {func_label} ({'VULNERABLE' if func_label else 'SAFE'})")
    print(f"  Vuln types:      {vuln_types}")
    print(f"  Code preview:    {ex['raw_lines'][0][:80].strip()}")
    print()

# Function level label distribution
print("=== FUNCTION-LEVEL DISTRIBUTION ===\n")
func_labels = [int(any(l == 1 for l in ex['label'])) for ex in train_data]
vuln_count = sum(func_labels)
safe_count = len(func_labels) - vuln_count

print(f"Total functions: {len(func_labels)}")
print(f"Vulnerable:      {vuln_count} ({vuln_count/len(func_labels)*100:.1f}%)")
print(f"Safe:            {safe_count} ({safe_count/len(func_labels)*100:.1f}%)")

# Vulnerability type distribution
print("\n=== VULNERABILITY TYPE DISTRIBUTION ===\n")
all_types = []
for ex in train_data:
    for t, l in zip(ex['type'], ex['label']):
        if l == 1:
            all_types.append(t)

from collections import Counter
type_counts = Counter(all_types)
for t, c in type_counts.most_common(20):
    print(f"  {t:<30}: {c}")

In [ ]:
import os

print(os.getcwd())

In [ ]:
import os

base_path = os.path.abspath("..")  # goes from notebooks → backend

cve_path = os.path.join(base_path, "data", "raw", "CVEfixes")

print("BASE:", base_path)
print("CVE PATH:", cve_path)
print("EXISTS:", os.path.exists(cve_path))
print("FILES:", os.listdir(cve_path) if os.path.exists(cve_path) else "NOT FOUND")

In [ ]:
from datasets import load_dataset

print("Loading VUDENC...")
vudenc = load_dataset("DetectVul/Vudenc")
print(vudenc)

# Check label and type structure
ex = vudenc['train'][0]
print(f"\nExample types: {set(ex['type'])}")
print(f"Example labels: {set(ex['label'])}")
print(f"Lines count: {len(ex['raw_lines'])}")

# Count function-level distributions
print("\nCounting vulnerability types...")
from collections import Counter
type_counter = Counter()

for ex in vudenc['train']:
    pairs = zip(ex['type'], ex['label'])
    for t, l in pairs:
        if l == 1:
            type_counter[t] += 1

print("\nVulnerable line types (top 20):")
for t, c in type_counter.most_common(20):
    print(f"  {t:<35}: {c}")

In [ ]:
# ── EXPERIMENT BLOCK A ───────────────────────────
# Title: Enhanced Feature Extractor for Real Code
# Purpose: Add 4 general features that fire on
#          real production code (PyCode Vul)
# Keep original extract_features() untouched above

def extract_features_v2(code: str) -> list:
    """
    Extended feature extractor — 16 features
    Original 12 + 4 new general features
    that work on real production code
    """
    # Get original 12 features first
    base_features = extract_features(code)

    try:
        tree = ast.parse(code)
    except SyntaxError:
        return base_features + [0, 0, 0, 0]

    # ── NEW FEATURE 13: User controlled input ────
    # Fires on request.args, request.form,
    # request.get, request.POST, request.data
    f13 = int(bool(
        re.search(r'request\.(args|form|get|POST|data|json)', code) or
        re.search(r'request\[', code) or
        re.search(r'flask\.request', code) or
        re.search(r'self\.(request|req)\.', code)
    ))

    # ── NEW FEATURE 14: Database operation ───────
    # Fires on any DB query/execute pattern
    f14 = int(bool(
        re.search(r'\.(execute|query|filter|raw|cursor)\s*\(', code) or
        re.search(r'(SELECT|INSERT|UPDATE|DELETE)', code, re.IGNORECASE) or
        re.search(r'db\.(session|execute|query)', code) or
        re.search(r'cursor\.(execute|fetchall|fetchone)', code)
    ))

    # ── NEW FEATURE 15: File/subprocess operation ─
    # Fires on file handling or process execution
    f15 = int(bool(
        re.search(r'\bopen\s*\(', code) or
        re.search(r'os\.(system|popen|makedirs|remove|rename)', code) or
        re.search(r'subprocess\.(call|run|Popen|check_output)', code) or
        re.search(r'shutil\.(copy|move|rmtree)', code)
    ))

    # ── NEW FEATURE 16: Dangerous deserialization ─
    # Fires on pickle, yaml.load, marshal, eval
    f16 = int(bool(
        re.search(r'pickle\.(loads|load|dumps)', code) or
        re.search(r'yaml\.load\s*\(', code) or
        re.search(r'marshal\.(loads|load)', code) or
        re.search(r'jsonpickle\.decode', code) or
        re.search(r'shelve\.open', code)
    ))

    return base_features + [f13, f14, f15, f16]


print("extract_features_v2 defined — 16 features")
print(f"Test: {len(extract_features_v2('x = 1'))} features returned")

In [ ]:
# ── EXPERIMENT BLOCK B ───────────────────────────
# Title: Sanity Check v2 on Synthetic + Real Data
# Purpose: Verify new features fire correctly

feature_names_v2 = [
    'f1_sql_concat', 'f2_hardcoded_secret', 'f3_eval_exec',
    'f4_path_traversal', 'f5_cmd_injection', 'f6_ast_nodes',
    'f7_string_count', 'f8_uses_environ', 'f9_parameterized',
    'f10_user_input', 'f11_ast_dangerous_calls',
    'f12_ast_hardcoded_assign',
    'f13_user_controlled_input', 'f14_db_operation',
    'f15_file_subprocess', 'f16_dangerous_deserialize'
]

print("=== SANITY CHECK — SYNTHETIC DATA (should be same as before) ===\n")
for i in range(3):
    row = df.iloc[i]
    feats = extract_features_v2(str(row['code']))
    print(f"Sample {i} (label={row['label']}):")
    print(f"  Original 12: {feats[:12]}")
    print(f"  New 4:       {feats[12:]}")
    print()

print("=== SANITY CHECK — PyCode Vul REAL CODE ===\n")

RAW_DATA = r"E:\Portfolio Project 2026\securescope-ai\backend\data\raw"
pycode_train = pd.read_excel(os.path.join(RAW_DATA, 'PyCode_Vul_train.xlsx'))
pycode_train = pycode_train[['vulnerable_function_source', 'label']]
pycode_train.columns = ['code', 'label']
pycode_train['code'] = pycode_train['code'].astype(str)

print(f"PyCode Vul loaded: {len(pycode_train)} rows\n")

# Check 5 vulnerable examples
vuln_samples = pycode_train[pycode_train['label']==1].head(5)
all_zero_count = 0

for i, row in vuln_samples.iterrows():
    feats = extract_features_v2(str(row['code']))
    is_all_zero_base = sum(feats[:12]) == 0
    any_new_fires = any(feats[12:])
    if is_all_zero_base:
        all_zero_count += 1
    print(f"Row {i} (vulnerable):")
    print(f"  Base features firing: {sum(feats[:12])}")
    print(f"  New features:         {feats[12:]}")
    print(f"  f13(request): {feats[12]} | f14(db): {feats[13]} | f15(file/proc): {feats[14]} | f16(deserial): {feats[15]}")
    print()

print(f"Rows with ALL base features = 0: {all_zero_count}/5")

In [ ]:
# ── EXPERIMENT BLOCK C ───────────────────────────
# Title: Full Feature Extraction on Combined Dataset
# Purpose: Run v2 extractor on PyCode Vul + synthetic
#          Compare firing rates to original

import numpy as np
import time

# Combine PyCode Vul + synthetic
pycode_sample = pycode_train.sample(n=2750, random_state=42)
combined_v2 = pd.concat([df[['code','label']], pycode_sample], ignore_index=True)
combined_v2['code'] = combined_v2['code'].astype(str)
combined_v2['label'] = combined_v2['label'].astype(int)

print(f"Combined dataset: {len(combined_v2)} rows")
print(f"Labels: {combined_v2['label'].value_counts().to_dict()}\n")

# Extract features
print("Extracting features...")
start = time.time()
rows_v2 = []
errors = 0

for idx, row in combined_v2.iterrows():
    try:
        feats = extract_features_v2(str(row['code']))
        feats.append(int(row['label']))
        rows_v2.append(feats)
    except Exception:
        errors += 1
        rows_v2.append([0]*16 + [int(row['label'])])

print(f"Done in {time.time()-start:.1f}s | Errors: {errors}")

X_v2 = pd.DataFrame(rows_v2, columns=feature_names_v2 + ['label'])

print(f"\nFeature matrix: {X_v2.shape}")
print(f"\nFeature firing rates:")
for col in feature_names_v2:
    rate = (X_v2[col] > 0).mean() * 100
    print(f"  {col:<35}: {rate:.1f}%")

print(f"\nAll-zero rows (excluding label): {(X_v2[feature_names_v2].sum(axis=1)==0).sum()}")

In [ ]:
# ── EXPERIMENT BLOCK D ───────────────────────────
# Title: Retrain ANN with v2 features + real data
# Purpose: Compare v2 model against original model

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
import tensorflow as tf
from tensorflow import keras
import joblib
import numpy as np

# ── Prepare data ──────────────────────────────────
feature_cols_v2 = [c for c in X_v2.columns if c != 'label']
X_train_v2 = X_v2[feature_cols_v2].values
y_train_v2 = X_v2['label'].values

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_v2, y_train_v2,
    test_size=0.15,
    random_state=42,
    stratify=y_train_v2
)

# ── Scale ─────────────────────────────────────────
scaler_v2 = StandardScaler()
X_tr_scaled  = scaler_v2.fit_transform(X_tr)
X_val_scaled = scaler_v2.transform(X_val)

print(f"Train: {X_tr_scaled.shape}")
print(f"Val:   {X_val_scaled.shape}")

# ── Build model ───────────────────────────────────
def build_ann_v2(input_dim):
    model = keras.Sequential([
        keras.layers.Input(shape=(input_dim,)),
        keras.layers.Dense(128, activation='relu'),
        keras.layers.BatchNormalization(),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(64, activation='relu'),
        keras.layers.BatchNormalization(),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(32, activation='relu'),
        keras.layers.Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy',
                 keras.metrics.AUC(name='auc'),
                 keras.metrics.Precision(name='precision'),
                 keras.metrics.Recall(name='recall')]
    )
    return model

model_v2 = build_ann_v2(16)
model_v2.summary()

# ── Train ─────────────────────────────────────────
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=7,
        min_lr=1e-6
    )
]

print("\nTraining v2 model...")
history_v2 = model_v2.fit(
    X_tr_scaled, y_tr,
    validation_data=(X_val_scaled, y_val),
    epochs=150,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

# ── Evaluate ──────────────────────────────────────
print("\n" + "="*40)
print("V2 MODEL RESULTS")
print("="*40)

y_pred_v2 = (model_v2.predict(X_val_scaled, verbose=0) > 0.5).astype(int).flatten()
print(f"\nValidation Accuracy: {accuracy_score(y_val, y_pred_v2)*100:.1f}%")
print(classification_report(y_val, y_pred_v2,
      target_names=['Safe', 'Vulnerable']))

# ── Save v2 model ─────────────────────────────────
MODELS_DIR = r"E:\Portfolio Project 2026\securescope-ai\backend\models\saved"

model_v2.save(os.path.join(MODELS_DIR, 'ann_v2.keras'))
joblib.dump(scaler_v2, os.path.join(MODELS_DIR, 'scaler_v2.pkl'))

print(f"\nModel saved: ann_v2.keras")
print(f"Scaler saved: scaler_v2.pkl")

In [ ]:
# ── EXPERIMENT BLOCK D Results ────────────────────
from sklearn.metrics import classification_report, accuracy_score

y_pred_v2 = (model_v2.predict(X_val_scaled, verbose=0) > 0.5).astype(int).flatten()

print("="*40)
print("V2 MODEL FINAL RESULTS")
print("="*40)
print(f"\nValidation Accuracy: {accuracy_score(y_val, y_pred_v2)*100:.1f}%")
print(f"\n{classification_report(y_val, y_pred_v2, target_names=['Safe', 'Vulnerable'])}")

# Compare with original model
print("="*40)
print("COMPARISON")
print("="*40)
print(f"Original model (synthetic only, 12 features): 96.0%")
print(f"V2 model (real + synthetic, 16 features):     {accuracy_score(y_val, y_pred_v2)*100:.1f}%")

In [ ]:
# ── EXPERIMENT BLOCK E ───────────────────────────
# Title: Threshold Tuning
# Purpose: Find the best threshold that maximizes
#          recall without destroying precision too much
# Default threshold is 0.5 — we try lower values

from sklearn.metrics import recall_score, precision_score, f1_score, accuracy_score
import numpy as np

print("=== THRESHOLD TUNING ===\n")
print(f"{'Threshold':<12} {'Accuracy':<12} {'Precision':<12} {'Recall':<12} {'F1':<12}")
print("-" * 60)

y_probs_v2 = model_v2.predict(X_val_scaled, verbose=0).flatten()

best_threshold = 0.5
best_f1 = 0

for threshold in [0.5, 0.45, 0.40, 0.35, 0.30, 0.25]:
    y_pred_t = (y_probs_v2 > threshold).astype(int)
    acc  = accuracy_score(y_val, y_pred_t)
    prec = precision_score(y_val, y_pred_t, zero_division=0)
    rec  = recall_score(y_val, y_pred_t)
    f1   = f1_score(y_val, y_pred_t)
    print(f"{threshold:<12} {acc:<12.3f} {prec:<12.3f} {rec:<12.3f} {f1:<12.3f}")
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold

print(f"\nBest threshold by F1: {best_threshold}")
print(f"Best F1 score: {best_f1:.3f}")

In [ ]:
# ── EXPERIMENT BLOCK F ───────────────────────────
# Title: Retrain with Class Weights
# Purpose: Tell the model that missing a vulnerable
#          example is 2x more costly than a false alarm
#          This pushes the model to improve recall

import tensorflow as tf
from tensorflow import keras

# Build fresh model — same architecture as v2
model_v2_weighted = keras.Sequential([
    keras.layers.Input(shape=(16,)),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid')
])

model_v2_weighted.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy',
             keras.metrics.AUC(name='auc'),
             keras.metrics.Precision(name='precision'),
             keras.metrics.Recall(name='recall')]
)

callbacks_w = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=7,
        min_lr=1e-6
    )
]

print("Training with class weights (vulnerable = 2x cost)...")
print("This tells the model: missing a vulnerability is worse than a false alarm\n")

history_weighted = model_v2_weighted.fit(
    X_tr_scaled, y_tr,
    validation_data=(X_val_scaled, y_val),
    epochs=150,
    batch_size=32,
    class_weight={0: 1.0, 1: 2.0},
    callbacks=callbacks_w,
    verbose=1
)

# Evaluate
print("\n" + "="*40)
print("WEIGHTED MODEL RESULTS")
print("="*40)

y_pred_w = (model_v2_weighted.predict(X_val_scaled, verbose=0) > 0.5).astype(int).flatten()
print(f"\nAccuracy: {accuracy_score(y_val, y_pred_w)*100:.1f}%")
print(classification_report(y_val, y_pred_w,
      target_names=['Safe', 'Vulnerable']))

In [ ]:
# ── EXPERIMENT BLOCK G ───────────────────────────
# Title: Final Comparison — All Models
# Purpose: Compare all 3 approaches side by side
#          Pick the best one for deployment

from sklearn.metrics import recall_score, precision_score, f1_score, accuracy_score

print("="*60)
print("FINAL MODEL COMPARISON")
print("="*60)
print(f"\n{'Model':<35} {'Accuracy':<10} {'Precision':<12} {'Recall':<10} {'F1':<8}")
print("-"*60)

# Original model — synthetic only (load from saved)
import joblib
import numpy as np

# V1 on validation set (approximate — different scaler)
# Just report known result
print(f"{'v1 synthetic (12 feat, synth only)':<35} {'96.0%':<10} {'0.98':<12} {'0.93':<10} {'0.95':<8}")

# V2 model — real + synthetic, default threshold
y_pred_v2_05 = (model_v2.predict(X_val_scaled, verbose=0).flatten() > 0.5).astype(int)
acc  = accuracy_score(y_val, y_pred_v2_05)
prec = precision_score(y_val, y_pred_v2_05)
rec  = recall_score(y_val, y_pred_v2_05)
f1   = f1_score(y_val, y_pred_v2_05)
print(f"{'v2 real+synth (16 feat, t=0.5)':<35} {acc*100:.1f}%{'':<5} {prec:<12.3f} {rec:<10.3f} {f1:<8.3f}")

# V2 model — best threshold from Block E
y_pred_v2_best = (model_v2.predict(X_val_scaled, verbose=0).flatten() > best_threshold).astype(int)
acc  = accuracy_score(y_val, y_pred_v2_best)
prec = precision_score(y_val, y_pred_v2_best)
rec  = recall_score(y_val, y_pred_v2_best)
f1   = f1_score(y_val, y_pred_v2_best)
print(f"{'v2 real+synth (16 feat, t={:.2f})':<35} {acc*100:.1f}%{'':<5} {prec:<12.3f} {rec:<10.3f} {f1:<8.3f}".format(best_threshold))

# Weighted model
y_pred_w = (model_v2_weighted.predict(X_val_scaled, verbose=0).flatten() > 0.5).astype(int)
acc  = accuracy_score(y_val, y_pred_w)
prec = precision_score(y_val, y_pred_w)
rec  = recall_score(y_val, y_pred_w)
f1   = f1_score(y_val, y_pred_w)
print(f"{'v2 weighted (class_weight=2.0)':<35} {acc*100:.1f}%{'':<5} {prec:<12.3f} {rec:<10.3f} {f1:<8.3f}")

print("\nFor a security tool — RECALL is the most important metric.")
print("Missing a vulnerability is worse than a false alarm.")
print("\nRecommended model for deployment: highest recall with precision > 0.75")

In [ ]:
pycode_test_raw = pd.read_csv(os.path.join(RAW_DATA, "PyCode_Vul_test.csv"))

print(pycode_test_raw.columns.tolist())
display(pycode_test_raw.head())

In [ ]:
# ── IMPROVEMENT BLOCK 1 ──────────────────────────
# Title: Enhanced Feature Extractor v3
# Purpose: Add 6 more meaningful features
#          Total will be 22 features

def extract_features_v3(code: str) -> list:
    """
    Full feature extractor — 22 features
    v2 base (16) + 6 new structural features
    """
    # Get v2 base features first
    base = extract_features_v2(code)

    try:
        tree = ast.parse(code)
    except SyntaxError:
        return base + [0, 0, 0, 0, 0, 0]

    # ── F17: Maximum nesting depth ────────────────
    # Deeply nested code = more complex = harder to audit
    # Count maximum indentation level in the code
    lines = code.split('\n')
    max_indent = 0
    for line in lines:
        if line.strip():
            indent = len(line) - len(line.lstrip())
            max_indent = max(max_indent, indent)
    f17 = max_indent // 4  # convert spaces to nesting level

    # ── F18: Function parameter count ────────────
    # More parameters = more attack surface
    f18 = 0
    for node in ast.walk(tree):
        if isinstance(node, ast.FunctionDef):
            f18 = max(f18, len(node.args.args))

    # ── F19: Exception handling present ──────────
    # try/except can hide vulnerability indicators
    # Also bare except is a bad practice signal
    f19 = 0
    for node in ast.walk(tree):
        if isinstance(node, ast.ExceptHandler):
            if node.type is None:
                f19 = 2  # bare except — worse signal
            else:
                f19 = max(f19, 1)

    # ── F20: String formatting operations ────────
    # .format(), f-strings, % formatting
    # These can be used to build dangerous strings
    f20 = int(bool(
        re.search(r'\.format\s*\(', code) or
        re.search(r'f["\'].*{.*}.*["\']', code) or
        re.search(r'%\s*["\(]', code)
    ))

    # ── F21: Network/HTTP calls ───────────────────
    # requests.get, urllib, httplib, socket
    f21 = int(bool(
        re.search(r'requests\.(get|post|put|delete|patch)', code) or
        re.search(r'urllib\.(request|urlopen)', code) or
        re.search(r'\bsocket\b', code) or
        re.search(r'http\.client', code) or
        re.search(r'httplib', code)
    ))

    # ── F22: Weak crypto / random usage ──────────
    # random.random() for security = bad
    # MD5/SHA1 for passwords = bad
    # hardcoded IVs = bad
    f22 = int(bool(
        re.search(r'\brandom\.(random|randint|choice)\b', code) or
        re.search(r'hashlib\.(md5|sha1)\b', code) or
        re.search(r'MD5|SHA1', code) or
        re.search(r'DES|RC4|ECB', code)
    ))

    return base + [f17, f18, f19, f20, f21, f22]


feature_names_v3 = feature_names_v2 + [
    'f17_nesting_depth',
    'f18_param_count',
    'f19_exception_handling',
    'f20_string_formatting',
    'f21_network_calls',
    'f22_weak_crypto'
]

print("extract_features_v3 defined — 22 features")
print(f"Test: {len(extract_features_v3('x = 1'))} features returned")

# Quick sanity check
test_code = """
def get_user(user_id):
    try:
        query = "SELECT * FROM users WHERE id = " + user_id
        return db.execute(query)
    except:
        pass
"""
feats = extract_features_v3(test_code)
print(f"\nTest on SQL injection example:")
for name, val in zip(feature_names_v3, feats):
    if val != 0:
        print(f"  {name}: {val}")

In [ ]:
# ── IMPROVEMENT BLOCK 2 ──────────────────────────
# Title: Full Dataset + Proper Evaluation Split
# Purpose: Use ALL 14,248 PyCode Vul rows
#          Use PyCode Vul TEST SET as holdout
#          Never mix test data into training

import pandas as pd
import numpy as np
import os
import time

RAW_DATA = r"E:\Portfolio Project 2026\securescope-ai\backend\data\raw"

# Load full PyCode Vul
print("Loading full PyCode Vul dataset...")
pycode_full = pd.read_excel(os.path.join(RAW_DATA, 'PyCode_Vul_train.xlsx'))
pycode_full = pycode_full[['vulnerable_function_source', 'label']]
pycode_full.columns = ['code', 'label']
pycode_full['code'] = pycode_full['code'].astype(str)
pycode_full['label'] = pycode_full['label'].astype(int)

# Load PyCode Vul test set — holdout only
pycode_test = pd.read_csv(os.path.join(RAW_DATA, 'PyCode_Vul_test.csv'))
pycode_test = pycode_test.rename(columns={
    'function_code': 'code',
    'class': 'label'
})
pycode_test = pycode_test[['code', 'label']]
pycode_test['code'] = pycode_test['code'].astype(str)
pycode_test['label'] = pycode_test['label'].astype(int)

print(f"PyCode Vul train: {len(pycode_full)} rows")
print(f"PyCode Vul test:  {len(pycode_test)} rows (holdout — never used in training)")

# Combine training data
combined_v3 = pd.concat([df[['code','label']], pycode_full], ignore_index=True)
combined_v3['code'] = combined_v3['code'].astype(str)
combined_v3['label'] = combined_v3['label'].astype(int)

print(f"\nCombined training data: {len(combined_v3)} rows")
print(f"Labels: {combined_v3['label'].value_counts().to_dict()}")

# Extract v3 features from training data
print("\nExtracting v3 features from training data...")
print("This will take 3-5 minutes...")
start = time.time()

rows_v3 = []
errors = 0
for idx, row in combined_v3.iterrows():
    try:
        feats = extract_features_v3(str(row['code']))
        feats.append(int(row['label']))
        rows_v3.append(feats)
    except Exception:
        errors += 1
        rows_v3.append([0]*22 + [int(row['label'])])

    if (idx + 1) % 3000 == 0:
        print(f"  {idx+1}/{len(combined_v3)} rows ({time.time()-start:.1f}s)")

print(f"Done in {time.time()-start:.1f}s | Errors: {errors}")

# Build feature dataframe
X_v3_df = pd.DataFrame(rows_v3, columns=feature_names_v3 + ['label'])

print(f"\nTraining feature matrix: {X_v3_df.shape}")
print(f"Labels: {X_v3_df['label'].value_counts().to_dict()}")

# Extract features from test set
print("\nExtracting v3 features from test set...")
test_rows = []
for idx, row in pycode_test.iterrows():
    try:
        feats = extract_features_v3(str(row['code']))
        feats.append(int(row['label']))
        test_rows.append(feats)
    except Exception:
        test_rows.append([0]*22 + [int(row['label'])])

X_test_df = pd.DataFrame(test_rows, columns=feature_names_v3 + ['label'])
print(f"Test feature matrix: {X_test_df.shape}")
print(f"Test labels: {X_test_df['label'].value_counts().to_dict()}")

# Feature firing rates
print("\nFeature firing rates on combined training data:")
for col in feature_names_v3:
    rate = (X_v3_df[col] > 0).mean() * 100
    print(f"  {col:<35}: {rate:.1f}%")

In [ ]:
%pip install tensorboard

In [ ]:
# ── IMPROVEMENT BLOCK 3 ──────────────────────────
# Title: Hyperparameter Tuning with Keras Tuner
# Purpose: Find best learning rate, layer sizes,
#          dropout rates automatically
# This is real ML engineering — not guessing

# Install first
import subprocess
subprocess.run(['pip', 'install', 'keras-tuner', '-q'])

import keras_tuner as kt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Prepare data
X_v3 = X_v3_df[feature_names_v3].values
y_v3 = X_v3_df['label'].values

X_tr3, X_val3, y_tr3, y_val3 = train_test_split(
    X_v3, y_v3,
    test_size=0.15,
    random_state=42,
    stratify=y_v3
)

scaler_v3 = StandardScaler()
X_tr3_scaled  = scaler_v3.fit_transform(X_tr3)
X_val3_scaled = scaler_v3.transform(X_val3)

print(f"Train: {X_tr3_scaled.shape}")
print(f"Val:   {X_val3_scaled.shape}")

# Define model builder for tuner
def build_tunable_model(hp):
    model = keras.Sequential()
    model.add(keras.layers.Input(shape=(22,)))

    # Tune number of layers (2 or 3)
    for i in range(hp.Int('num_layers', 2, 3)):
        model.add(keras.layers.Dense(
            units=hp.Choice(f'units_{i}', [64, 128, 256]),
            activation='relu'
        ))
        model.add(keras.layers.BatchNormalization())
        model.add(keras.layers.Dropout(
            rate=hp.Float(f'dropout_{i}', 0.1, 0.5, step=0.1)
        ))

    model.add(keras.layers.Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=hp.Choice('lr', [0.001, 0.0005, 0.0001])
        ),
        loss='binary_crossentropy',
        metrics=['accuracy',
                 keras.metrics.AUC(name='auc'),
                 keras.metrics.Precision(name='precision'),
                 keras.metrics.Recall(name='recall')]
    )
    return model

# Run tuner
print("\nStarting hyperparameter search...")
print("This will try multiple configurations — takes 15-30 minutes")
print("Each trial trains a different model configuration\n")

tuner = kt.BayesianOptimization(
    build_tunable_model,
    objective=kt.Objective('val_auc', direction='max'),
    max_trials=15,
    num_initial_points=5,
    directory='kt_results',
    project_name='securescope_v3',
    overwrite=True
)

tuner.search(
    X_tr3_scaled, y_tr3,
    validation_data=(X_val3_scaled, y_val3),
    epochs=50,
    batch_size=32,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor='val_auc',
            patience=10,
            mode='max'
        )
    ],
    verbose=0
)

# Get best hyperparameters
best_hps = tuner.get_best_hyperparameters(1)[0]
print("\n=== BEST HYPERPARAMETERS FOUND ===")
print(f"Number of layers: {best_hps.get('num_layers')}")
for i in range(best_hps.get('num_layers')):
    print(f"Layer {i} units:   {best_hps.get(f'units_{i}')}")
    print(f"Layer {i} dropout: {best_hps.get(f'dropout_{i}')}")
print(f"Learning rate:    {best_hps.get('lr')}")

# Train best model fully
print("\nTraining best model with full epochs...")
best_model = tuner.hypermodel.build(best_hps)

history_best = best_model.fit(
    X_tr3_scaled, y_tr3,
    validation_data=(X_val3_scaled, y_val3),
    epochs=150,
    batch_size=32,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=15,
            restore_best_weights=True
        )
    ],
    verbose=1
)

# Evaluate tuned model
from sklearn.metrics import classification_report, accuracy_score

y_pred_tuned = (best_model.predict(X_val3_scaled, verbose=0) > 0.4).astype(int).flatten()
print("\n=== TUNED MODEL RESULTS (threshold=0.40) ===")
print(f"Accuracy: {accuracy_score(y_val3, y_pred_tuned)*100:.1f}%")
print(classification_report(y_val3, y_pred_tuned,
      target_names=['Safe', 'Vulnerable']))

In [ ]:
# ── IMPROVEMENT BLOCK 4 ──────────────────────────
# Title: Ensemble — 3 Models Combined
# Purpose: Train 3 different models and combine
#          their predictions by voting
#          Ensemble almost always beats single model
# Three approaches combined:
#   Model A: ANN (neural network)
#   Model B: Random Forest (tree-based)
#   Model C: XGBoost (gradient boosting)

from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

print("Training ensemble of 3 models...\n")

# Model A — Random Forest
print("Training Random Forest...")
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=5,
    class_weight={0: 1.0, 1: 1.5},
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_tr3_scaled, y_tr3)
rf_pred = rf_model.predict(X_val3_scaled)
rf_proba = rf_model.predict_proba(X_val3_scaled)[:, 1]
print(f"  Random Forest accuracy: {accuracy_score(y_val3, rf_pred)*100:.1f}%")

# Model B — Gradient Boosting
print("Training Gradient Boosting...")
gb_model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    random_state=42
)
gb_model.fit(X_tr3_scaled, y_tr3)
gb_pred = gb_model.predict(X_val3_scaled)
gb_proba = gb_model.predict_proba(X_val3_scaled)[:, 1]
print(f"  Gradient Boosting accuracy: {accuracy_score(y_val3, gb_pred)*100:.1f}%")

# Model C — Best ANN from tuner
ann_proba = best_model.predict(X_val3_scaled, verbose=0).flatten()
ann_pred  = (ann_proba > 0.4).astype(int)
print(f"  ANN (tuned) accuracy: {accuracy_score(y_val3, ann_pred)*100:.1f}%")

# ── Soft Voting Ensemble ──────────────────────────
# Average probabilities from all 3 models
# Each model votes with its confidence level
print("\n=== SOFT VOTING ENSEMBLE ===")
print("Averaging probabilities from RF + GB + ANN\n")

ensemble_proba = (rf_proba + gb_proba + ann_proba) / 3

# Try thresholds on ensemble
print(f"{'Threshold':<12} {'Accuracy':<12} {'Precision':<12} {'Recall':<12} {'F1':<12}")
print("-"*60)

best_ensemble_threshold = 0.5
best_ensemble_f1 = 0

for threshold in [0.5, 0.45, 0.40, 0.35, 0.30]:
    from sklearn.metrics import precision_score, recall_score, f1_score
    y_ens = (ensemble_proba > threshold).astype(int)
    acc  = accuracy_score(y_val3, y_ens)
    prec = precision_score(y_val3, y_ens, zero_division=0)
    rec  = recall_score(y_val3, y_ens)
    f1   = f1_score(y_val3, y_ens)
    print(f"{threshold:<12} {acc:<12.3f} {prec:<12.3f} {rec:<12.3f} {f1:<12.3f}")
    if f1 > best_ensemble_f1:
        best_ensemble_f1 = f1
        best_ensemble_threshold = threshold

print(f"\nBest ensemble threshold: {best_ensemble_threshold}")
print(f"Best ensemble F1: {best_ensemble_f1:.3f}")

# Final ensemble report
y_ens_best = (ensemble_proba > best_ensemble_threshold).astype(int)
print(f"\n=== ENSEMBLE FINAL REPORT (threshold={best_ensemble_threshold}) ===")
print(classification_report(y_val3, y_ens_best,
      target_names=['Safe', 'Vulnerable']))

# Feature importance from Random Forest
print("\n=== FEATURE IMPORTANCE (Random Forest) ===")
importances = rf_model.feature_importances_
feat_imp = sorted(zip(feature_names_v3, importances),
                  key=lambda x: x[1], reverse=True)
for feat, imp in feat_imp[:10]:
    bar = '█' * int(imp * 100)
    print(f"  {feat:<35}: {imp:.4f} {bar}")

In [ ]:
# ── IMPROVEMENT BLOCK 5 ──────────────────────────
# Title: Complete Model Comparison
# Purpose: Show full progression from v1 to ensemble
#          Evaluate ALL models on real holdout test set

from sklearn.metrics import precision_score, recall_score, f1_score

print("="*65)
print("COMPLETE MODEL PROGRESSION — REAL TEST SET EVALUATION")
print("="*65)

# Prepare real holdout test set
X_holdout = X_test_df[feature_names_v3].values
y_holdout  = X_test_df['label'].values
X_holdout_scaled = scaler_v3.transform(X_holdout)

print(f"\nHoldout test set: {len(y_holdout)} real PyCode Vul examples")
print(f"Labels: {dict(zip(*np.unique(y_holdout, return_counts=True)))}\n")

print(f"{'Model':<40} {'Acc':<8} {'Prec':<8} {'Rec':<8} {'F1':<8}")
print("-"*65)

# V1 synthetic baseline — report known numbers
print(f"{'v1: ANN synthetic (12 feat)':<40} {'~96%':<8} {'0.98':<8} {'0.93':<8} {'0.95':<8}")
print(f"{'    NOTE: tested on synthetic data only':<40}")
print()

# V2 on holdout
ann_v2_holdout = scaler_v2.transform(
    X_test_df[feature_names_v2].values
)
y_v2_h = (model_v2.predict(ann_v2_holdout, verbose=0).flatten() > 0.4).astype(int)
print(f"{'v2: ANN real+synth (16 feat, t=0.4)':<40} "
      f"{accuracy_score(y_holdout, y_v2_h)*100:.1f}%{'':<3} "
      f"{precision_score(y_holdout, y_v2_h):.3f}{'':<3} "
      f"{recall_score(y_holdout, y_v2_h):.3f}{'':<3} "
      f"{f1_score(y_holdout, y_v2_h):.3f}")

# Tuned ANN on holdout
y_tuned_h = (best_model.predict(X_holdout_scaled, verbose=0).flatten() > 0.4).astype(int)
print(f"{'v3: ANN tuned (22 feat, HPT, t=0.4)':<40} "
      f"{accuracy_score(y_holdout, y_tuned_h)*100:.1f}%{'':<3} "
      f"{precision_score(y_holdout, y_tuned_h):.3f}{'':<3} "
      f"{recall_score(y_holdout, y_tuned_h):.3f}{'':<3} "
      f"{f1_score(y_holdout, y_tuned_h):.3f}")

# RF on holdout
y_rf_h = rf_model.predict(X_holdout_scaled)
print(f"{'v3: Random Forest (22 feat)':<40} "
      f"{accuracy_score(y_holdout, y_rf_h)*100:.1f}%{'':<3} "
      f"{precision_score(y_holdout, y_rf_h):.3f}{'':<3} "
      f"{recall_score(y_holdout, y_rf_h):.3f}{'':<3} "
      f"{f1_score(y_holdout, y_rf_h):.3f}")

# GB on holdout
y_gb_h = gb_model.predict(X_holdout_scaled)
print(f"{'v3: Gradient Boosting (22 feat)':<40} "
      f"{accuracy_score(y_holdout, y_gb_h)*100:.1f}%{'':<3} "
      f"{precision_score(y_holdout, y_gb_h):.3f}{'':<3} "
      f"{recall_score(y_holdout, y_gb_h):.3f}{'':<3} "
      f"{f1_score(y_holdout, y_gb_h):.3f}")

# Ensemble on holdout
rf_h_proba  = rf_model.predict_proba(X_holdout_scaled)[:, 1]
gb_h_proba  = gb_model.predict_proba(X_holdout_scaled)[:, 1]
ann_h_proba = best_model.predict(X_holdout_scaled, verbose=0).flatten()
ens_h_proba = (rf_h_proba + gb_h_proba + ann_h_proba) / 3
y_ens_h = (ens_h_proba > best_ensemble_threshold).astype(int)
print(f"{'v3: ENSEMBLE (ANN+RF+GB, 22 feat)':<40} "
      f"{accuracy_score(y_holdout, y_ens_h)*100:.1f}%{'':<3} "
      f"{precision_score(y_holdout, y_ens_h):.3f}{'':<3} "
      f"{recall_score(y_holdout, y_ens_h):.3f}{'':<3} "
      f"{f1_score(y_holdout, y_ens_h):.3f}")

print("\nBest model for deployment: highest F1 with recall > 0.75")

In [ ]:
# ── IMPROVEMENT BLOCK 6 ──────────────────────────
# Title: Save Best Model for Deployment
# Purpose: Save whichever model won the comparison

import joblib
import json

MODELS_DIR = r"E:\Portfolio Project 2026\securescope-ai\backend\models\saved"

# Save all models — keep everything
best_model.save(os.path.join(MODELS_DIR, 'ann_v3_tuned.keras'))
joblib.dump(rf_model,  os.path.join(MODELS_DIR, 'rf_v3.pkl'))
joblib.dump(gb_model,  os.path.join(MODELS_DIR, 'gb_v3.pkl'))
joblib.dump(scaler_v3, os.path.join(MODELS_DIR, 'scaler_v3.pkl'))

# Save ensemble config
ensemble_config = {
    "version": "v3_ensemble",
    "models": ["ann_v3_tuned.keras", "rf_v3.pkl", "gb_v3.pkl"],
    "voting": "soft",
    "threshold": best_ensemble_threshold,
    "n_features": 22,
    "feature_names": feature_names_v3,
    "training_data": "2750 synthetic + 14248 PyCode Vul real functions",
    "test_set": "PyCode Vul test set (3563 real functions, never seen in training)"
}

with open(os.path.join(MODELS_DIR, 'ensemble_config.json'), 'w') as f:
    json.dump(ensemble_config, f, indent=2)

print("=== ALL MODELS SAVED ===")
print(f"ann_v3_tuned.keras  ← best neural network")
print(f"rf_v3.pkl           ← random forest")
print(f"gb_v3.pkl           ← gradient boosting")
print(f"scaler_v3.pkl       ← feature scaler")
print(f"ensemble_config.json ← deployment configuration")

In [ ]:
# ── EDA BLOCK 1 ───────────────────────────────────
# Title: Dataset Overview
# Purpose: Understand basic structure of all data
#          before touching any features

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.style.use('dark_background')
sns.set_palette("husl")

DOCS_DIR = r"E:\Portfolio Project 2026\securescope-ai\backend\docs"
RAW_DATA = r"E:\Portfolio Project 2026\securescope-ai\backend\data\raw"
os.makedirs(DOCS_DIR, exist_ok=True)

print("=" * 55)
print("EDA — SECURESCOPE AI DATASET ANALYSIS")
print("=" * 55)

# ── Synthetic dataset overview ────────────────────
print("\n=== SYNTHETIC DATASET ===")
print(f"Total examples:   {len(df)}")
print(f"Vulnerable:       {(df['label']==1).sum()}")
print(f"Safe:             {(df['label']==0).sum()}")
print(f"Type distribution:\n{df['type'].value_counts()}")
print(f"Missing values:\n{df.isnull().sum()}")
print(f"Duplicate rows:   {df.duplicated(subset='code').sum()}")

# ── PyCode Vul overview ───────────────────────────
print("\n=== PYCODE VUL DATASET ===")
print(f"Train rows:       {len(pycode_full)}")
print(f"Test rows:        {len(pycode_test)}")
print(f"Train labels:\n{pycode_full['label'].value_counts()}")
print(f"Test labels:\n{pycode_test['label'].value_counts()}")
print(f"Train missing:\n{pycode_full.isnull().sum()}")
print(f"Train duplicates: {pycode_full.duplicated(subset='code').sum()}")

# ── Combined dataset overview ─────────────────────
print("\n=== COMBINED TRAINING DATA ===")
print(f"Total rows:       {len(combined_v3)}")
print(f"Vulnerable:       {(combined_v3['label']==1).sum()}")
print(f"Safe:             {(combined_v3['label']==0).sum()}")
print(f"Imbalance ratio:  "
      f"{(combined_v3['label']==0).sum() / (combined_v3['label']==1).sum():.2f}:1")

In [ ]:
# ── EDA BLOCK 2 ───────────────────────────────────
# Title: Class Distribution Visualization
# Purpose: See class balance clearly in charts

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Class Distribution Across Datasets', fontsize=14)

# Synthetic dataset
syn_counts = df['label'].value_counts()
axes[0].bar(['Safe', 'Vulnerable'],
            [syn_counts[0], syn_counts[1]],
            color=['#2ecc71', '#e74c3c'], alpha=0.8)
axes[0].set_title('Synthetic Dataset\n(2,750 examples)')
axes[0].set_ylabel('Count')
for i, v in enumerate([syn_counts[0], syn_counts[1]]):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold')

# PyCode Vul
pyc_counts = pycode_full['label'].value_counts()
axes[1].bar(['Safe', 'Vulnerable'],
            [pyc_counts[0], pyc_counts[1]],
            color=['#2ecc71', '#e74c3c'], alpha=0.8)
axes[1].set_title('PyCode Vul Train\n(14,248 examples)')
for i, v in enumerate([pyc_counts[0], pyc_counts[1]]):
    axes[1].text(i, v + 50, str(v), ha='center', fontweight='bold')

# Combined
comb_counts = combined_v3['label'].value_counts()
axes[2].bar(['Safe', 'Vulnerable'],
            [comb_counts[0], comb_counts[1]],
            color=['#2ecc71', '#e74c3c'], alpha=0.8)
axes[2].set_title('Combined Training\n(~17,000 examples)')
for i, v in enumerate([comb_counts[0], comb_counts[1]]):
    axes[2].text(i, v + 100, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(DOCS_DIR, 'eda_class_distribution.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: eda_class_distribution.png")

In [ ]:
# ── EDA BLOCK 3 ───────────────────────────────────
# Title: Code Length Analysis
# Purpose: Understand how long functions are
#          Short vs long code behaves differently
#          Outliers can hurt model training

# Compute code lengths
df['code_length']          = df['code'].apply(lambda x: len(str(x)))
df['line_count']           = df['code'].apply(lambda x: len(str(x).split('\n')))
pycode_full['code_length'] = pycode_full['code'].apply(lambda x: len(str(x)))
pycode_full['line_count']  = pycode_full['code'].apply(lambda x: len(str(x).split('\n')))

print("=== CODE LENGTH STATISTICS ===\n")
print("Synthetic dataset:")
print(df.groupby('label')[['code_length', 'line_count']].describe().round(1))
print("\nPyCode Vul:")
print(pycode_full.groupby('label')[['code_length', 'line_count']].describe().round(1))

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Code Length Distribution', fontsize=14)

# Synthetic — line count
axes[0][0].hist(
    [df[df['label']==0]['line_count'],
     df[df['label']==1]['line_count']],
    bins=30, label=['Safe', 'Vulnerable'],
    color=['#2ecc71', '#e74c3c'], alpha=0.7
)
axes[0][0].set_title('Synthetic — Line Count')
axes[0][0].set_xlabel('Lines of code')
axes[0][0].legend()

# Synthetic — char length
axes[0][1].hist(
    [df[df['label']==0]['code_length'],
     df[df['label']==1]['code_length']],
    bins=30, label=['Safe', 'Vulnerable'],
    color=['#2ecc71', '#e74c3c'], alpha=0.7
)
axes[0][1].set_title('Synthetic — Character Length')
axes[0][1].set_xlabel('Characters')
axes[0][1].legend()

# PyCode Vul — line count (cap at 200 for visibility)
pyc_safe = pycode_full[pycode_full['label']==0]['line_count'].clip(upper=200)
pyc_vuln = pycode_full[pycode_full['label']==1]['line_count'].clip(upper=200)
axes[1][0].hist(
    [pyc_safe, pyc_vuln],
    bins=40, label=['Safe', 'Vulnerable'],
    color=['#2ecc71', '#e74c3c'], alpha=0.7
)
axes[1][0].set_title('PyCode Vul — Line Count (capped at 200)')
axes[1][0].set_xlabel('Lines of code')
axes[1][0].legend()

# PyCode Vul — char length (cap at 5000)
pyc_safe_c = pycode_full[pycode_full['label']==0]['code_length'].clip(upper=5000)
pyc_vuln_c = pycode_full[pycode_full['label']==1]['code_length'].clip(upper=5000)
axes[1][1].hist(
    [pyc_safe_c, pyc_vuln_c],
    bins=40, label=['Safe', 'Vulnerable'],
    color=['#2ecc71', '#e74c3c'], alpha=0.7
)
axes[1][1].set_title('PyCode Vul — Char Length (capped at 5000)')
axes[1][1].set_xlabel('Characters')
axes[1][1].legend()

plt.tight_layout()
plt.savefig(os.path.join(DOCS_DIR, 'eda_code_length.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: eda_code_length.png")

# Outlier detection
p99_lines = pycode_full['line_count'].quantile(0.99)
p99_chars = pycode_full['code_length'].quantile(0.99)
print(f"\nOutlier thresholds (99th percentile):")
print(f"  Line count:  {p99_lines:.0f} lines")
print(f"  Char length: {p99_chars:.0f} chars")
print(f"Rows above line threshold: "
      f"{(pycode_full['line_count'] > p99_lines).sum()}")
print(f"Rows above char threshold: "
      f"{(pycode_full['code_length'] > p99_chars).sum()}")

In [ ]:
# ── EDA BLOCK 4 ───────────────────────────────────
# Title: Data Cleaning
# Purpose: Remove data-quality problems identified in EDA
#          Nulls, empty code, duplicates, invalid Python
#          Report long-code outliers without deleting them

print("=== DATA CLEANING ===\n")

initial_count = len(combined_v3)
print(f"Starting rows: {initial_count}")

# Step 1 — Remove null code
combined_v3 = combined_v3.dropna(subset=["code"])
print(
    f"After null removal:       {len(combined_v3)} "
    f"(removed {initial_count - len(combined_v3)})"
)

# Step 2 — Ensure code is string
combined_v3["code"] = combined_v3["code"].astype(str)

# Step 3 — Remove empty / tiny code
before_empty = len(combined_v3)
combined_v3 = combined_v3[combined_v3["code"].str.strip().str.len() > 10]
print(
    f"After empty removal:      {len(combined_v3)} "
    f"(removed {before_empty - len(combined_v3)})"
)

# Step 4 — Remove exact duplicate code
before_dedup = len(combined_v3)
combined_v3 = combined_v3.drop_duplicates(subset="code")
print(
    f"After deduplication:      {len(combined_v3)} "
    f"(removed {before_dedup - len(combined_v3)})"
)

# Step 5 — Report long-code outliers, do not remove
combined_v3["line_count"] = combined_v3["code"].apply(
    lambda x: len(str(x).split("\n"))
)
combined_v3["code_length"] = combined_v3["code"].apply(
    lambda x: len(str(x))
)

line_threshold = combined_v3["line_count"].quantile(0.99)
char_threshold = combined_v3["code_length"].quantile(0.99)

long_code_mask = (
    (combined_v3["line_count"] > line_threshold) |
    (combined_v3["code_length"] > char_threshold)
)

long_code_count = long_code_mask.sum()

print("\nLong-code outlier report only:")
print(f"99th percentile line count:  {line_threshold:.0f}")
print(f"99th percentile char length: {char_threshold:.0f}")
print(f"Long-code rows found:        {long_code_count}")
print("Long-code label distribution:")
print(combined_v3.loc[long_code_mask, "label"].value_counts())

# Step 6 — Verify syntax parseable
import ast as ast_module

def is_valid_python(code):
    try:
        ast_module.parse(str(code))
        return True
    except Exception:
        return False

print("\nChecking Python syntax validity...")
combined_v3["valid_syntax"] = combined_v3["code"].apply(is_valid_python)

invalid_count = (~combined_v3["valid_syntax"]).sum()
invalid_labels = combined_v3.loc[~combined_v3["valid_syntax"], "label"].value_counts()

print(f"Invalid Python rows:      {invalid_count}")
print("Invalid label distribution:")
print(invalid_labels)

combined_v3 = combined_v3[combined_v3["valid_syntax"]]

# Step 7 — Cleanup helper columns
combined_v3 = combined_v3.drop(
    columns=["line_count", "code_length", "valid_syntax"],
    errors="ignore"
)
combined_v3 = combined_v3.reset_index(drop=True)

print(f"\n=== CLEANING SUMMARY ===")
print(f"Started with:  {initial_count} rows")
print(f"Finished with: {len(combined_v3)} rows")
print(
    f"Removed:       {initial_count - len(combined_v3)} rows "
    f"({(initial_count - len(combined_v3)) / initial_count * 100:.1f}%)"
)

print(f"\nFinal label distribution:")
print(combined_v3["label"].value_counts())

safe_count = combined_v3[combined_v3["label"] == 0].shape[0]
vuln_count = combined_v3[combined_v3["label"] == 1].shape[0]

print(f"Imbalance ratio: {safe_count / vuln_count:.2f}:1")

In [ ]:
# ── EDA BLOCK 5 ───────────────────────────────────
# Title: Feature Distribution Analysis
# Purpose: After extraction, analyze each feature
#          Distributions, correlations, class separation

# This block runs AFTER feature extraction (Block 2)
# Uses X_v3_df which has all 22 features + label

print("=== FEATURE DISTRIBUTION ANALYSIS ===\n")

feature_cols = feature_names_v3

# ── 1. Feature firing rates by class ─────────────
print("Feature firing rates — Vulnerable vs Safe:\n")
print(f"{'Feature':<35} {'Vuln %':<10} {'Safe %':<10} {'Separation':<10}")
print("-" * 65)

vuln_df = X_v3_df[X_v3_df['label'] == 1]
safe_df = X_v3_df[X_v3_df['label'] == 0]

separations = []
for col in feature_cols:
    vuln_rate = (vuln_df[col] > 0).mean() * 100
    safe_rate = (safe_df[col] > 0).mean() * 100
    separation = vuln_rate - safe_rate
    separations.append((col, vuln_rate, safe_rate, separation))
    marker = " ★" if abs(separation) > 10 else ""
    print(f"{col:<35} {vuln_rate:<10.1f} {safe_rate:<10.1f} "
          f"{separation:<10.1f}{marker}")

print("\n★ = strong separator (>10% difference)")

# ── 2. Feature separation bar chart ──────────────
separations_sorted = sorted(separations, key=lambda x: abs(x[3]), reverse=True)
feat_names_sorted  = [s[0] for s in separations_sorted]
sep_values         = [s[3] for s in separations_sorted]
colors             = ['#e74c3c' if v > 0 else '#2ecc71' for v in sep_values]

plt.figure(figsize=(14, 8))
bars = plt.barh(range(len(feat_names_sorted)), sep_values, color=colors, alpha=0.8)
plt.yticks(range(len(feat_names_sorted)), feat_names_sorted)
plt.xlabel('Separation Score (Vulnerable % - Safe %)')
plt.title('Feature Separation Power\n'
          'Red = fires more on vulnerable | Green = fires more on safe')
plt.axvline(x=0, color='white', linewidth=0.5)
plt.tight_layout()
plt.savefig(os.path.join(DOCS_DIR, 'eda_feature_separation.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: eda_feature_separation.png")

In [ ]:
# ── EDA BLOCK 6 ───────────────────────────────────
# Title: Correlation Analysis

print("=== CORRELATION ANALYSIS ===\n")

constant_features = [
    c for c in feature_cols
    if X_v3_df[c].nunique(dropna=False) <= 1
]

corr_features = [c for c in feature_cols if c not in constant_features]

if constant_features:
    print("Constant features skipped from correlation:")
    print(constant_features)

corr_matrix = X_v3_df[corr_features].corr()

plt.figure(figsize=(16, 12))
mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask)] = True

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="RdYlGn",
    center=0,
    square=True,
    linewidths=0.5,
    annot_kws={"size": 7}
)

plt.title(
    "Feature Correlation Matrix\n"
    "High correlation = potentially redundant features"
)
plt.tight_layout()
plt.savefig(os.path.join(DOCS_DIR, "eda_correlation_matrix.png"),
            dpi=150, bbox_inches="tight")
plt.show()

print("Saved: eda_correlation_matrix.png")

print("\nHighly correlated feature pairs (|correlation| > 0.7):")
high_corr_pairs = []

for i in range(len(corr_features)):
    for j in range(i + 1, len(corr_features)):
        corr_val = corr_matrix.iloc[i, j]
        if pd.notna(corr_val) and abs(corr_val) > 0.7:
            high_corr_pairs.append(
                (corr_features[i], corr_features[j], corr_val)
            )
            print(f"  {corr_features[i]} <-> {corr_features[j]}: {corr_val:.3f}")

if not high_corr_pairs:
    print("  None found — all features are sufficiently independent")
else:
    print("\n  Consider removing one from each highly correlated pair")

print("\nCorrelation of each feature with label (target):")
label_corr = X_v3_df[corr_features + ["label"]].corr()["label"].drop("label")
label_corr_sorted = label_corr.reindex(
    label_corr.abs().sort_values(ascending=False).index
)

for feat, corr_val in label_corr_sorted.items():
    direction = "+" if corr_val >= 0 else "-"
    bar = "█" * int(abs(corr_val) * 30)
    print(f"  {feat:<35}: {direction}{abs(corr_val):.4f} {bar}")

In [ ]:
# ── EDA BLOCK 7 ───────────────────────────────────
# Title: Feature Distribution Boxplots
# Purpose: See how continuous features (f6, f7, f17,
#          f18) differ between vulnerable and safe

continuous_features = [
    'f6_ast_nodes', 'f7_string_count',
    'f17_nesting_depth', 'f18_param_count'
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Continuous Feature Distributions\n'
             'Vulnerable vs Safe', fontsize=13)

for idx, feat in enumerate(continuous_features):
    ax = axes[idx // 2][idx % 2]

    vuln_vals = X_v3_df[X_v3_df['label']==1][feat].clip(upper=100)
    safe_vals = X_v3_df[X_v3_df['label']==0][feat].clip(upper=100)

    ax.boxplot(
        [safe_vals, vuln_vals],
        labels=['Safe', 'Vulnerable'],
        patch_artist=True,
        boxprops=dict(facecolor='#2ecc71', alpha=0.7),
        medianprops=dict(color='white', linewidth=2)
    )
    ax.set_title(f'{feat}')
    ax.set_ylabel('Value (capped at 100)')

    # Add mean markers
    ax.plot(1, safe_vals.mean(), 'wo', markersize=8, label=f'mean={safe_vals.mean():.1f}')
    ax.plot(2, vuln_vals.mean(), 'ro', markersize=8, label=f'mean={vuln_vals.mean():.1f}')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(DOCS_DIR, 'eda_feature_boxplots.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: eda_feature_boxplots.png")

In [ ]:
# ── EDA BLOCK 8 ───────────────────────────────────
# Title: EDA Summary Report

print("=" * 60)
print("EDA COMPLETE — SUMMARY REPORT")
print("=" * 60)

synth_rows = len(df)
pycode_rows = len(pycode_full)
combined_rows = len(combined_v3)

safe_count = int((combined_v3["label"] == 0).sum())
vuln_count = int((combined_v3["label"] == 1).sum())
imbalance_ratio = safe_count / vuln_count

line_counts = combined_v3["code"].astype(str).apply(lambda x: len(x.split("\n")))
char_lengths = combined_v3["code"].astype(str).apply(len)

line_p99 = line_counts.quantile(0.99)
char_p99 = char_lengths.quantile(0.99)

long_mask = (line_counts > line_p99) | (char_lengths > char_p99)
long_count = int(long_mask.sum())

# Synthetic duplicate check
synth_duplicates = int(df.duplicated(subset="code").sum())

# Strong feature separators from EDA Block 5
if "feature_sep_df" in globals():
    strong_features = feature_sep_df[
        feature_sep_df["abs_separation"] > 10
    ]["feature"].tolist()
else:
    strong_features = []

strong_text = (
    ", ".join(strong_features)
    if strong_features
    else "None above 10 percentage points"
)

# Highly correlated pairs from EDA Block 6
high_corr_count = len(high_corr_pairs) if "high_corr_pairs" in globals() else 0

# Cleaning counts
removed_rows = 16998 - combined_rows  # original combined total from EDA Block 1

print(f"""
DATASET QUALITY:
  Synthetic:  {synth_rows:,} examples, balanced source dataset
  Synthetic duplicate code rows found: {synth_duplicates}
  PyCode Vul: {pycode_rows:,} train examples, real GitHub code
  Combined:   {combined_rows:,} rows after cleaning ({removed_rows:,} removed)

CLASS BALANCE:
  Safe:       {safe_count:,}
  Vulnerable: {vuln_count:,}
  Safe/Vulnerable ratio: {imbalance_ratio:.2f}:1
  Mild imbalance — acceptable for baseline training

CODE LENGTH:
  Synthetic:  short controlled functions
  PyCode Vul: realistic long-tail function lengths
  Outliers:   long-code outliers identified and retained
  Current p99 line count:  {line_p99:.0f}
  Current p99 char length: {char_p99:.0f}
  Long-code rows retained: {long_count:,}

DATA CLEANING PERFORMED:
  1. Null code values removed
  2. Empty/too-short code checked (<10 chars)
  3. Exact duplicate code rows removed
  4. Long-code outliers reported, not removed
  5. Invalid Python syntax rows removed

FEATURE QUALITY:
  Strong separators: {strong_text}
  Redundant features: checked via correlation matrix
  Highly correlated pairs > 0.7: {high_corr_count}

GRAPHS SAVED TO docs/:
  eda_class_distribution.png
  eda_code_length.png
  eda_feature_separation.png
  eda_correlation_matrix.png
  eda_feature_boxplots.png

READY FOR:
  Model training on cleaned feature matrix
  Threshold tuning
  Validation and holdout evaluation
""")

In [ ]:
# ── IMPROVEMENT BLOCK 3 UPDATED ──────────────────
# Title: XGBoost + LightGBM + ANN Ensemble
# Purpose: Replace RF with better tree methods
#          Feature importance analysis included

# Install once if needed
import sys
import subprocess
subprocess.run(
    [sys.executable, "-m", "pip", "install", "xgboost", "lightgbm", "-q"],
    check=True
)

import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import classification_report, accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np
import matplotlib.pyplot as plt

# Class imbalance from cleaned data
scale_pos = (y_tr3 == 0).sum() / (y_tr3 == 1).sum()
print(f"Using scale_pos_weight/class weight: {scale_pos:.2f}")

# Tree models do not require scaled features
X_tr_tree = X_tr3
X_val_tree = X_val3

# ── Model A: XGBoost ──────────────────────────────
print("Training XGBoost...")

xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_tr_tree, y_tr3,
    eval_set=[(X_val_tree, y_val3)],
    verbose=False
)

xgb_pred = xgb_model.predict(X_val_tree)
xgb_proba = xgb_model.predict_proba(X_val_tree)[:, 1]

print(f"  XGBoost accuracy:  {accuracy_score(y_val3, xgb_pred) * 100:.1f}%")
print(f"  XGBoost precision: {precision_score(y_val3, xgb_pred):.3f}")
print(f"  XGBoost recall:    {recall_score(y_val3, xgb_pred):.3f}")
print(f"  XGBoost F1:        {f1_score(y_val3, xgb_pred):.3f}")

# ── Model B: LightGBM ─────────────────────────────
print("\nTraining LightGBM...")

lgb_model = lgb.LGBMClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight={0: 1.0, 1: scale_pos},
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

lgb_model.fit(
    X_tr_tree, y_tr3,
    eval_set=[(X_val_tree, y_val3)]
)

lgb_pred = lgb_model.predict(X_val_tree)
lgb_proba = lgb_model.predict_proba(X_val_tree)[:, 1]

print(f"  LightGBM accuracy:  {accuracy_score(y_val3, lgb_pred) * 100:.1f}%")
print(f"  LightGBM precision: {precision_score(y_val3, lgb_pred):.3f}")
print(f"  LightGBM recall:    {recall_score(y_val3, lgb_pred):.3f}")
print(f"  LightGBM F1:        {f1_score(y_val3, lgb_pred):.3f}")

# ── Model C: Tuned ANN ────────────────────────────
print("\nEvaluating tuned ANN...")

ann_proba = best_model.predict(X_val3_scaled, verbose=0).flatten()
ann_pred = (ann_proba > 0.4).astype(int)

print(f"  ANN accuracy:  {accuracy_score(y_val3, ann_pred) * 100:.1f}%")
print(f"  ANN precision: {precision_score(y_val3, ann_pred):.3f}")
print(f"  ANN recall:    {recall_score(y_val3, ann_pred):.3f}")
print(f"  ANN F1:        {f1_score(y_val3, ann_pred):.3f}")

In [ ]:
# ── IMPROVEMENT BLOCK 4 UPDATED ──────────────────
# Title: Feature Importance Analysis
# Purpose: Understand which features matter most
#          Remove noise, keep signal

print("=== FEATURE IMPORTANCE ANALYSIS ===\n")

# Use same class weight as current cleaned split
scale_pos = (y_tr3 == 0).sum() / (y_tr3 == 1).sum()
print(f"Using scale_pos_weight/class weight: {scale_pos:.2f}\n")

# XGBoost feature importance is normalized
xgb_importance = xgb_model.feature_importances_

# LightGBM split/gain importance is not on same scale by default
lgb_importance_raw = lgb_model.feature_importances_.astype(float)
lgb_importance = (
    lgb_importance_raw / lgb_importance_raw.sum()
    if lgb_importance_raw.sum() > 0
    else lgb_importance_raw
)

# Average normalized importance across both tree models
avg_importance = (xgb_importance + lgb_importance) / 2

feat_imp_pairs = sorted(
    zip(feature_names_v3, xgb_importance, lgb_importance, avg_importance),
    key=lambda x: x[3],
    reverse=True
)

print(f"{'Feature':<35} {'XGBoost':<10} {'LightGBM':<10} {'Average':<10}")
print("-" * 75)

for feat, xgb_imp, lgb_imp, avg_imp in feat_imp_pairs:
    bar = "█" * int(avg_imp * 100)
    print(
        f"{feat:<35} {xgb_imp:<10.4f} "
        f"{lgb_imp:<10.4f} {avg_imp:<10.4f} {bar}"
    )

# Identify weak features
importance_threshold = 0.02

print(f"\n=== WEAK FEATURES (avg importance < {importance_threshold}) ===")
weak_features = [f for f, _, _, avg in feat_imp_pairs if avg < importance_threshold]
strong_features_tree = [f for f, _, _, avg in feat_imp_pairs if avg >= importance_threshold]

print(f"Weak:   {weak_features}")
print(f"Strong: {len(strong_features_tree)} features carrying the weight")

# Retrain XGBoost on strong features only
print(f"\nRetraining XGBoost on {len(strong_features_tree)} strong features only...")

strong_idx = [feature_names_v3.index(f) for f in strong_features_tree]

# Tree models use unscaled features
X_tr_strong = X_tr3[:, strong_idx]
X_val_strong = X_val3[:, strong_idx]

xgb_lean = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_lean.fit(X_tr_strong, y_tr3, verbose=False)

xgb_lean_pred = xgb_lean.predict(X_val_strong)
xgb_lean_proba = xgb_lean.predict_proba(X_val_strong)[:, 1]

xgb_full_f1 = f1_score(y_val3, xgb_pred)
xgb_lean_f1 = f1_score(y_val3, xgb_lean_pred)

print(f"XGBoost full accuracy: {accuracy_score(y_val3, xgb_pred) * 100:.1f}%")
print(f"XGBoost full F1:       {xgb_full_f1:.3f}")
print(f"XGBoost full recall:   {recall_score(y_val3, xgb_pred):.3f}")

print(f"\nXGBoost lean accuracy: {accuracy_score(y_val3, xgb_lean_pred) * 100:.1f}%")
print(f"XGBoost lean F1:       {xgb_lean_f1:.3f}")
print(f"XGBoost lean recall:   {recall_score(y_val3, xgb_lean_pred):.3f}")

print(
    f"\nDid removing weak features help? "
    f"{'Yes' if xgb_lean_f1 > xgb_full_f1 else 'No — keep all 22'}"
)

In [ ]:
# ── IMPROVEMENT BLOCK 5 UPDATED ──────────────────
# Title: Soft Voting Ensemble — Final
# Purpose: Combine ANN + XGBoost + LightGBM
#          Average probabilities, find best threshold

print("=== SOFT VOTING ENSEMBLE ===\n")
print("Combining: ANN (neural) + XGBoost (boosting) + LightGBM (boosting)")
print("Method: average probability scores from all 3 models")
print("Selection rule: highest F1 with recall > 0.75\n")

ensemble_proba = (ann_proba + xgb_proba + lgb_proba) / 3

thresholds = np.arange(0.20, 0.61, 0.01)

results = []

print(
    f"{'Threshold':<12} {'Accuracy':<12} {'Precision':<12} "
    f"{'Recall':<12} {'F1':<12} {'Eligible':<10}"
)
print("-" * 75)

for threshold in thresholds:
    y_ens = (ensemble_proba >= threshold).astype(int)

    acc = accuracy_score(y_val3, y_ens)
    prec = precision_score(y_val3, y_ens, zero_division=0)
    rec = recall_score(y_val3, y_ens, zero_division=0)
    f1 = f1_score(y_val3, y_ens, zero_division=0)
    eligible = rec > 0.75

    results.append({
        "threshold": threshold,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "eligible": eligible
    })

results_df = pd.DataFrame(results)

eligible_df = results_df[results_df["eligible"]].copy()

if len(eligible_df) > 0:
    best_row = eligible_df.sort_values("f1", ascending=False).iloc[0]
else:
    best_row = results_df.sort_values("f1", ascending=False).iloc[0]

best_threshold = best_row["threshold"]
best_f1 = best_row["f1"]

for _, row in results_df.iterrows():
    marker = ""
    if abs(row["threshold"] - best_threshold) < 1e-9:
        marker = " <- selected"

    # Print every 0.05 threshold plus selected threshold
    is_major_tick = int(round(row["threshold"] * 100)) % 5 == 0
    if is_major_tick or marker:
        print(
            f"{row['threshold']:<12.2f} "
            f"{row['accuracy']:<12.3f} "
            f"{row['precision']:<12.3f} "
            f"{row['recall']:<12.3f} "
            f"{row['f1']:<12.3f} "
            f"{str(row['eligible']):<10}"
            f"{marker}"
        )

if len(eligible_df) > 0:
    print("\nSelected threshold uses deployment rule: highest F1 with recall > 0.75")
else:
    print("\nNo threshold met recall > 0.75; selected highest F1 overall")

print(f"Best threshold: {best_threshold:.2f}")
print(f"Best F1:        {best_f1:.3f}")
print(f"Recall:         {best_row['recall']:.3f}")
print(f"Precision:      {best_row['precision']:.3f}")
print(f"Accuracy:       {best_row['accuracy'] * 100:.1f}%")

y_ens_best = (ensemble_proba >= best_threshold).astype(int)

print("\n=== ENSEMBLE FINAL REPORT ===")
print(classification_report(
    y_val3,
    y_ens_best,
    target_names=["Safe", "Vulnerable"]
))

In [ ]:
# ── IMPROVEMENT BLOCK 6 UPDATED ──────────────────
# Title: Complete Comparison Table
# Purpose: Every model evaluated on real holdout test
#          Honest final numbers

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import numpy as np

print("=" * 70)
print("FINAL MODEL COMPARISON — REAL HOLDOUT TEST SET")
print("=" * 70)

X_holdout = X_test_df[feature_names_v3].values
y_holdout = X_test_df["label"].values.astype(int)

# ANN needs scaled features
X_hold_scaled = scaler_v3.transform(X_holdout)

# Tree models were trained on unscaled features
X_hold_tree = X_holdout

print(f"Holdout rows: {len(y_holdout):,}")
print(f"Safe:         {(y_holdout == 0).sum():,}")
print(f"Vulnerable:   {(y_holdout == 1).sum():,}")

def get_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0)
    }

rows = []

def add_result(name, y_pred, notes=""):
    m = get_metrics(y_holdout, y_pred)
    rows.append({
        "model": name,
        "accuracy": m["accuracy"],
        "precision": m["precision"],
        "recall": m["recall"],
        "f1": m["f1"],
        "eligible": m["recall"] > 0.75,
        "notes": notes
    })

# V1 baseline — synthetic-only, not comparable for deployment
rows.append({
    "model": "v1: ANN synthetic only (12 feat)",
    "accuracy": np.nan,
    "precision": np.nan,
    "recall": np.nan,
    "f1": np.nan,
    "eligible": False,
    "notes": "not comparable: synthetic-only evaluation was inflated"
})

# V2 ANN
ann_v2_h = scaler_v2.transform(X_test_df[feature_names_v2].values)
y_v2_h = (model_v2.predict(ann_v2_h, verbose=0).flatten() >= 0.40).astype(int)
add_result("v2: ANN real+synth (16 feat, t=0.40)", y_v2_h)

# V3 ANN tuned
ann_h_proba = best_model.predict(X_hold_scaled, verbose=0).flatten()
y_ann_h = (ann_h_proba >= 0.40).astype(int)
add_result("v3: ANN tuned HPT (22 feat, t=0.40)", y_ann_h)

# XGBoost full
y_xgb_h = xgb_model.predict(X_hold_tree)
xgb_h_proba = xgb_model.predict_proba(X_hold_tree)[:, 1]
add_result("v3: XGBoost (22 feat)", y_xgb_h)

# LightGBM full
y_lgb_h = lgb_model.predict(X_hold_tree)
lgb_h_proba = lgb_model.predict_proba(X_hold_tree)[:, 1]
add_result("v3: LightGBM (22 feat)", y_lgb_h)

# XGBoost lean
if "xgb_lean" in globals() and "strong_idx" in globals():
    X_hold_strong = X_hold_tree[:, strong_idx]
    y_xgb_lean_h = xgb_lean.predict(X_hold_strong)
    add_result("v3: XGBoost lean (strong feat only)", y_xgb_lean_h)

# Ensemble
ens_h_proba = (xgb_h_proba + lgb_h_proba + ann_h_proba) / 3
y_ens_h = (ens_h_proba >= best_threshold).astype(int)

add_result(
    f"v3: ENSEMBLE ANN+XGB+LGB (t={best_threshold:.2f})",
    y_ens_h
)

results_table = pd.DataFrame(rows)

print(f"\n{'Model':<46} {'Acc':<8} {'Prec':<8} {'Rec':<8} {'F1':<8} {'Eligible':<9}")
print("-" * 95)

for _, row in results_table.iterrows():
    if pd.isna(row["f1"]):
        print(
            f"{row['model']:<46} {'N/A':<8} {'N/A':<8} "
            f"{'N/A':<8} {'N/A':<8} {'False':<9}"
        )
    else:
        print(
            f"{row['model']:<46} "
            f"{row['accuracy'] * 100:<7.1f}% "
            f"{row['precision']:<8.3f} "
            f"{row['recall']:<8.3f} "
            f"{row['f1']:<8.3f} "
            f"{str(row['eligible']):<9}"
        )

valid_results = results_table.dropna(subset=["f1"]).copy()
eligible_results = valid_results[valid_results["eligible"]].copy()

print(f"\n{'=' * 70}")

if len(eligible_results) > 0:
    best_row = eligible_results.sort_values("f1", ascending=False).iloc[0]
    print("Selection rule: highest F1 with recall > 0.75")
else:
    best_row = valid_results.sort_values("f1", ascending=False).iloc[0]
    print("No model met recall > 0.75; selected highest F1 overall")

print(f"Best model: {best_row['model']}")
print(f"Accuracy:   {best_row['accuracy'] * 100:.1f}%")
print(f"Precision:  {best_row['precision']:.3f}")
print(f"Recall:     {best_row['recall']:.3f}")
print(f"F1:         {best_row['f1']:.3f}")

In [ ]:
# ── PHASE 1 GATE TUNING ──────────────────────────
# Goal: high-recall vulnerable/safe gate for ANN -> LSTM cascade
# Selection rule: highest F1 where recall >= target

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import numpy as np
import pandas as pd

print("=== PHASE 1 HIGH-RECALL GATE TUNING ===\n")

recall_target = 0.80

# Equal soft-voting ensemble from validation probabilities
ensemble_val_proba = (ann_proba + xgb_proba + lgb_proba) / 3

threshold_results = []

for threshold in np.arange(0.15, 0.61, 0.01):
    y_pred = (ensemble_val_proba >= threshold).astype(int)

    threshold_results.append({
        "threshold": round(float(threshold), 2),
        "accuracy": accuracy_score(y_val3, y_pred),
        "precision": precision_score(y_val3, y_pred, zero_division=0),
        "recall": recall_score(y_val3, y_pred, zero_division=0),
        "f1": f1_score(y_val3, y_pred, zero_division=0)
    })

gate_df = pd.DataFrame(threshold_results)
gate_df["eligible"] = gate_df["recall"] >= recall_target

eligible_gate_df = gate_df[gate_df["eligible"]]

if len(eligible_gate_df) > 0:
    best_gate = eligible_gate_df.sort_values("f1", ascending=False).iloc[0]
    print(f"Selected threshold by validation: highest F1 with recall >= {recall_target}")
else:
    best_gate = gate_df.sort_values("f1", ascending=False).iloc[0]
    print(f"No threshold reached recall >= {recall_target}; selected highest F1")

gate_threshold = float(best_gate["threshold"])

print("\nTop validation thresholds:")
display(
    gate_df.sort_values(["eligible", "f1"], ascending=[False, False])
           .head(10)
)

print("\nSelected gate threshold:")
print(best_gate)

In [ ]:
print(len(ann_proba), len(xgb_proba), len(lgb_proba), len(y_val3))

In [ ]:
# ── HOLDOUT EVALUATION FOR HIGH-RECALL GATE ──────

print("=== HOLDOUT EVALUATION — HIGH-RECALL GATE ===\n")

# Make sure holdout probabilities exist
X_holdout = X_test_df[feature_names_v3].values
y_holdout = X_test_df["label"].values.astype(int)
X_hold_scaled = scaler_v3.transform(X_holdout)

# ANN uses scaled features
ann_h_proba = best_model.predict(X_hold_scaled, verbose=0).flatten()

# XGBoost/LightGBM were trained on unscaled features
xgb_h_proba = xgb_model.predict_proba(X_holdout)[:, 1]
lgb_h_proba = lgb_model.predict_proba(X_holdout)[:, 1]

ensemble_hold_proba = (ann_h_proba + xgb_h_proba + lgb_h_proba) / 3

y_gate_holdout = (ensemble_hold_proba >= gate_threshold).astype(int)

gate_holdout_metrics = {
    "accuracy": accuracy_score(y_holdout, y_gate_holdout),
    "precision": precision_score(y_holdout, y_gate_holdout, zero_division=0),
    "recall": recall_score(y_holdout, y_gate_holdout, zero_division=0),
    "f1": f1_score(y_holdout, y_gate_holdout, zero_division=0)
}

print(f"Gate threshold: {gate_threshold:.2f}")
print(f"Accuracy:       {gate_holdout_metrics['accuracy'] * 100:.1f}%")
print(f"Precision:      {gate_holdout_metrics['precision']:.3f}")
print(f"Recall:         {gate_holdout_metrics['recall']:.3f}")
print(f"F1:             {gate_holdout_metrics['f1']:.3f}")

print("\nClassification report:")
print(classification_report(
    y_holdout,
    y_gate_holdout,
    target_names=["Safe", "Vulnerable"]
))

if gate_holdout_metrics["recall"] >= recall_target:
    print("\nResult: gate recall target met on holdout.")
else:
    print("\nResult: gate recall target NOT met on holdout.")

In [ ]:
# ── BLOCK 7 — Save Final Phase 1 Models ──────────
# Title: Save all models + config for deployment
# Purpose: Everything FastAPI needs is saved here

import joblib
import json
import os
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

MODELS_DIR = Path(r"E:\Portfolio Project 2026\securescope-ai\backend\models\saved")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# ── Save models ───────────────────────────────────
best_model.save(MODELS_DIR / "ann_v3_tuned.keras")
joblib.dump(xgb_model,  MODELS_DIR / "xgb_v3.pkl")
joblib.dump(lgb_model,  MODELS_DIR / "lgb_v3.pkl")
joblib.dump(scaler_v3,  MODELS_DIR / "scaler_v3.pkl")
print("Models saved.")

# ── Save deployment config ────────────────────────
config = {
    "version": "v3_ensemble_phase1",
    "description": "Phase 1 ANN gate — ensemble of ANN + XGBoost + LightGBM",

    "deployment": {
        "threshold": gate_threshold,
        "voting": "soft_average",
        "ann_uses_scaled_features": True,
        "tree_models_use_raw_features": True
    },

    "models": {
        "ann":      "ann_v3_tuned.keras",
        "xgboost":  "xgb_v3.pkl",
        "lightgbm": "lgb_v3.pkl",
        "scaler":   "scaler_v3.pkl"
    },

    "features": {
        "count": 22,
        "names": list(feature_names_v3)
    },

    "training_data": {
        "synthetic": {
            "rows": int(len(df)),
            "description": "Hand-crafted OWASP-aligned examples, 5 vulnerability types"
        },
        "real": {
            "rows": int(len(pycode_full)),
            "source": "PyCode Vul — real GitHub functions from Django, Flask, Airflow etc",
            "citation": "Karim et al. 2025, IEEE"
        },
        "combined_after_cleaning": int(len(combined_v3))
    },

    "evaluation": {
        "test_set": "PyCode Vul holdout — 3,563 real functions, never seen in training",
        "test_rows": int(len(y_holdout)),
        "safe_rows": int((y_holdout == 0).sum()),
        "vulnerable_rows": int((y_holdout == 1).sum()),
        "metrics": {
            "accuracy":  round(gate_holdout_metrics["accuracy"],  3),
            "precision": round(gate_holdout_metrics["precision"], 3),
            "recall":    round(gate_holdout_metrics["recall"],    3),
            "f1":        round(gate_holdout_metrics["f1"],        3)
        }
    },

    "known_limitations": [
        "Features are hand-crafted regex + AST patterns",
        "Model learns code complexity as proxy for vulnerability",
        "Recall ceiling ~0.77 due to feature representation limits",
        "Phase 2 BiLSTM on token sequences designed to address this"
    ],

    "next_phase": "Phase 2 — BiLSTM on tokenized code sequences"
}

config_path = MODELS_DIR / "ensemble_config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

# ── Print summary ─────────────────────────────────
print("\n" + "="*50)
print("PHASE 1 COMPLETE — FINAL SUMMARY")
print("="*50)
print(f"\nModels saved to: {MODELS_DIR}")
print(f"\nFiles:")
print(f"  ann_v3_tuned.keras  ← neural network")
print(f"  xgb_v3.pkl          ← XGBoost")
print(f"  lgb_v3.pkl          ← LightGBM")
print(f"  scaler_v3.pkl       ← StandardScaler")
print(f"  ensemble_config.json ← deployment config")

print(f"\nDeployment settings:")
print(f"  Threshold:   {gate_threshold:.2f}")
print(f"  Features:    22")
print(f"  Voting:      soft average")

print(f"\nHoldout test results (3,563 real functions):")
print(f"  Accuracy:    {gate_holdout_metrics['accuracy']*100:.1f}%")
print(f"  Precision:   {gate_holdout_metrics['precision']:.3f}")
print(f"  Recall:      {gate_holdout_metrics['recall']:.3f}")
print(f"  F1:          {gate_holdout_metrics['f1']:.3f}")

print(f"\nPhase 1 ML is complete.")
print(f"Next: FastAPI /scan endpoint")